# Bab 7: Membangun Aplikasi Obrolan
## Mulai Cepat API OpenAI

Notebook ini diadaptasi dari [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang mencakup notebook yang mengakses layanan [Azure OpenAI](aoai-assignment.ipynb).

API OpenAI Python juga bekerja dengan Model Azure OpenAI, dengan beberapa modifikasi. Pelajari lebih lanjut tentang perbedaannya di sini: [Cara beralih antara endpoint OpenAI dan Azure OpenAI dengan Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Ikhtisar  
"Model bahasa besar adalah fungsi yang memetakan teks ke teks. Diberikan sebuah string teks sebagai input, model bahasa besar mencoba memprediksi teks yang akan muncul berikutnya"(1). Notebook "quickstart" ini akan memperkenalkan pengguna pada konsep LLM tingkat tinggi, persyaratan paket inti untuk memulai dengan AML, pengenalan ringan tentang desain prompt, dan beberapa contoh singkat dari berbagai kasus penggunaan. 


## Daftar Isi  

[Ikhtisar](#overview)  
[Cara menggunakan Layanan OpenAI](#how-to-use-openai-service)  
[1. Membuat Layanan OpenAI Anda](#1.-creating-your-openai-service)  
[2. Instalasi](#2.-installation)    
[3. Kredensial](#3.-credentials)  

[Kasus Penggunaan](#use-cases)    
[1. Ringkas Teks](#1.-summarize-text)  
[2. Klasifikasikan Teks](#2.-classify-text)  
[3. Buat Nama Produk Baru](#3.-generate-new-product-names)  
[4. Menyempurnakan Klasifikator](#4.fine-tune-a-classifier)  

[Referensi](#references)


### Bangun prompt pertamamu  
Latihan singkat ini akan memberikan pengenalan dasar untuk mengirimkan prompt ke model OpenAI untuk tugas sederhana "ringkasan".


**Langkah-langkah**:  
1. Pasang library OpenAI di lingkungan python kamu  
2. Muat library pembantu standar dan atur kredensial keamanan OpenAI khas kamu untuk Layanan OpenAI yang telah kamu buat  
3. Pilih model untuk tugasmu  
4. Buat prompt sederhana untuk model  
5. Kirimkan permintaanmu ke API model!


### 1. Pasang OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Impor pustaka pembantu dan buat kredensial


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Menemukan model yang tepat  
Model seperti GPT-4o dan GPT-4o mini dapat memahami dan menghasilkan bahasa alami, dan tersedia di platform OpenAI dengan tingkat daya dan kecepatan yang berbeda yang sesuai untuk berbagai tugas.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Desain Prompt  

"Keajaiban model bahasa besar adalah bahwa dengan dilatih untuk meminimalkan kesalahan prediksi ini atas jumlah teks yang sangat besar, model akhirnya mempelajari konsep yang berguna untuk prediksi tersebut. Misalnya, mereka mempelajari konsep seperti"(1):

* cara mengeja
* cara kerja tata bahasa
* cara memparafrasekan
* cara menjawab pertanyaan
* cara mengadakan percakapan
* cara menulis dalam banyak bahasa
* cara membuat kode
* dll.

#### Cara mengendalikan model bahasa besar  
"Dari semua masukan ke model bahasa besar, yang paling berpengaruh adalah prompt teks(1).

Model bahasa besar dapat dipicu untuk menghasilkan keluaran dengan beberapa cara:

Instruksi: Beri tahu model apa yang Anda inginkan
Penyelesaian: Menginduksi model untuk menyelesaikan awal dari apa yang Anda inginkan
Demonstrasi: Tunjukkan pada model apa yang Anda inginkan, dengan salah satu cara:
Beberapa contoh dalam prompt
Ratusan atau ribuan contoh dalam dataset pelatihan fine-tuning"



#### Ada tiga pedoman dasar untuk membuat prompt:

**Tunjukkan dan jelaskan**. Buatlah jelas apa yang Anda inginkan baik melalui instruksi, contoh, atau kombinasi keduanya. Jika Anda ingin model mengurutkan daftar item secara alfabetis atau mengklasifikasikan paragraf berdasarkan sentimen, tunjukkan bahwa itu yang Anda inginkan.

**Berikan data berkualitas**. Jika Anda mencoba membangun klasifikasi atau membuat model mengikuti pola, pastikan ada cukup contoh. Pastikan untuk memeriksa kembali contoh Anda — model biasanya cukup pintar untuk melihat kesalahan ejaan dasar dan memberi respons, tetapi juga bisa menganggap ini disengaja dan ini bisa memengaruhi respons.

**Periksa pengaturan Anda.** Pengaturan temperature dan top_p mengendalikan seberapa deterministik model dalam menghasilkan respons. Jika Anda meminta respons di mana hanya ada satu jawaban yang benar, maka Anda ingin mengatur ini lebih rendah. Jika Anda menginginkan respons yang lebih beragam, maka Anda mungkin ingin mengaturnya lebih tinggi. Kesalahan nomor satu yang dilakukan orang dengan pengaturan ini adalah mengira ini adalah kendali "kecerdasan" atau "kreativitas".


Sumber: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Kirim!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Ringkas Teks  
#### Tantangan  
Ringkas teks dengan menambahkan 'tl;dr:' di akhir sebuah bagian teks. Perhatikan bagaimana model memahami cara melakukan sejumlah tugas tanpa instruksi tambahan. Anda dapat bereksperimen dengan perintah yang lebih deskriptif daripada tl;dr untuk memodifikasi perilaku model dan menyesuaikan ringkasan yang Anda terima(3).  

Pekerjaan terbaru telah menunjukkan peningkatan substansial pada banyak tugas dan tolok ukur NLP dengan melakukan pelatihan awal pada korpus teks yang besar diikuti oleh penyetelan khusus tugas. Meskipun biasanya arsitektur tidak tergantung pada tugas, metode ini masih memerlukan dataset penyetelan khusus tugas yang berisi ribuan atau puluhan ribu contoh. Sebaliknya, manusia umumnya dapat melakukan tugas bahasa baru hanya dari beberapa contoh atau instruksi sederhana - sesuatu yang masih sangat sulit dilakukan oleh sistem NLP saat ini. Di sini kami menunjukkan bahwa peningkatan skala model bahasa secara signifikan meningkatkan performa tanpa tergantung tugas dari beberapa contoh, terkadang bahkan mencapai kompetisi dengan pendekatan penyetelan khusus tugas yang sebelumnya terdepan. 



Tl;dr


# Latihan untuk beberapa kasus penggunaan  
1. Meringkas Teks  
2. Mengklasifikasikan Teks  
3. Menghasilkan Nama Produk Baru


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasifikasikan Teks  
#### Tantangan  
Klasifikasikan item ke dalam kategori yang disediakan pada saat inferensi. Dalam contoh berikut, kami menyediakan kategori dan teks yang akan diklasifikasikan dalam prompt(*playground_reference). 

Pertanyaan Pelanggan: Halo, salah satu tombol pada keyboard laptop saya baru-baru ini rusak dan saya membutuhkan penggantinya:

Kategori yang diklasifikasikan:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Hasilkan Nama Produk Baru
#### Tantangan
Buat nama produk dari kata contoh. Di sini kami menyertakan dalam prompt informasi tentang produk yang akan kami buat namanya. Kami juga memberikan contoh serupa untuk menunjukkan pola yang ingin kami terima. Kami juga telah menetapkan nilai temperature tinggi untuk meningkatkan keacakan dan respons yang lebih inovatif.

Deskripsi produk: Pembuat milkshake rumah
Kata benih: cepat, sehat, ringkas.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Deskripsi produk: Sepasang sepatu yang dapat muat di semua ukuran kaki.
Kata benih: dapat disesuaikan, pas, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referensi  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Praktik terbaik untuk penyetelan halus model GPT dalam mengklasifikasikan teks](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Untuk Bantuan Lebih Lanjut  
[Tim Komersialisasi OpenAI](AzureOpenAITeam@microsoft.com) 


# Kontributor
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
